# Stage 1 (GPU part): rollout corpus generation

Runs on a free Kaggle GPU. Before executing anything:

1. **Settings (right sidebar) -> Accelerator -> GPU T4 x2** (or P100).
2. **Settings -> Internet -> On** (needed to clone the repo and download models).
3. To run unattended (survives closing the browser tab): use **Save Version -> Save & Run All (Commit)** instead of running cells interactively. Free quota is ~30 GPU-hours/week; a session caps out around 9-12 hours.

This notebook: (a) clones the repo, (b) runs the CPU-only `--dry-run` smoke test first (catches repo/env problems for free, before touching the GPU or downloading any model), (c) installs the GPU extras, (d) runs a tiny real-GPU sanity check on 3 problems, (e) only then runs the full MATH500 128-problem x 32-rollout corpus generation.

Everything written under the cloned repo directory (which lives under `/kaggle/working/`) persists in the notebook's Output after a commit -- no extra copy step needed.

Resumable: each problem's rows are checkpointed to disk as soon as that problem finishes (not batched into one write at the end). If a session drops mid-run, just re-run the same full-run cell -- it automatically picks up where it left off instead of starting over. Pass `--fresh` to ignore an existing checkpoint and start clean.

In [ ]:
!git clone https://github.com/qqdexqq/twisted-smc-llm.git
%cd twisted-smc-llm

In [ ]:
# CPU-only deps first (fast, small). Kaggle's base image already has
# pandas/numpy/etc; -e . picks up anything missing (pyarrow, math-verify, datasets).
!pip install -q -e .

In [ ]:
# Free correctness check before spending any GPU time or downloading any
# model weights -- same philosophy as Stage 0. If this fails, the problem
# is in the repo/environment, not in a model or the GPU.
!python scripts/generate_rollouts.py --manifest manifests/math500_128.jsonl --dry-run --n 4

## GPU extras

Installs `torch`/`transformers`/`vllm`/`bitsandbytes`. **This is the one step that hasn't been tested end to end** (no GPU on the machine this repo was built on) -- vLLM installs are notorious for pulling in a different torch/CUDA build than the one Kaggle ships with. If the next cell errors on a CUDA/torch mismatch, the standard fix on Kaggle/Colab is: **Run -> Restart Session**, then re-run from this cell (skip the git clone / CPU-deps cells above).

In [ ]:
!pip install -q -e ".[gpu]"

In [ ]:
# Tiny real-GPU sanity check: 3 problems, N=4 rollouts, PRM in 8-bit
# (needed to fit the 7B PRM alongside the 1.5B generator on a 16GB T4/P100
# -- see models/prm.py). Eyeball the printed pass@1 -- it should be well
# above 0 on MATH500 with a real model (unlike the --dry-run mock, which
# is always ~0 by construction). If it's still ~0, something real is
# wrong (wrong chat template, PRM scoring convention, etc.) -- stop and
# debug before running the full batch.
!head -n 3 manifests/math500_128.jsonl > /tmp/math500_sanity_3.jsonl
!python scripts/generate_rollouts.py --manifest /tmp/math500_sanity_3.jsonl \
    --dataset-name math500_sanity3 \
    --generator Qwen/Qwen2.5-1.5B-Instruct --prm Qwen/Qwen2.5-Math-PRM-7B \
    --prm-type qwen --prm-8bit --n 4 --temperature 0.8 --max-tokens 1024 --seed 0

## Full run

Only run this once the sanity check above looks right. 128 problems x 32 rollouts on a 1.5B model is the Stage 1 dev-scale target from the plan doc (§5.3) -- expect this to take a while; keep an eye on the free-tier session time limit.

If this cell gets interrupted (session drop, disconnect, anything), just re-run it as-is -- it checkpoints progress per-problem and will print `checkpoint found: X/128 problems already complete, Y remaining` and only process what's left, rather than starting the whole 128x32 run over.

In [ ]:
!python scripts/generate_rollouts.py --manifest manifests/math500_128.jsonl \
    --generator Qwen/Qwen2.5-1.5B-Instruct --prm Qwen/Qwen2.5-Math-PRM-7B \
    --prm-type qwen --prm-8bit --n 32 --temperature 0.8 --max-tokens 2048 --seed 0

## After this finishes

- The printed pass@1 / best-of-32 / self-consistency numbers are the plan doc's "offline baselines for free" (§5.3) -- compare pass@1 against published MATH500 numbers for Qwen2.5-1.5B-Instruct as a rough correctness check on this whole pipeline.
- Parquet output (plus the raw `steps.checkpoint.jsonl` / `rollouts.checkpoint.jsonl` files it was built from) is under `data/rollouts/math500_128/.../seed=0/` -- `data/` is gitignored on purpose (plan doc §11: belongs on a persistent volume, not in git). Commit this notebook's **Output** (via Save & Run All) or download the parquet files directly from Kaggle's Output tab if you want them off of Kaggle.
- Next per the plan doc: Stage 2 -- reproduce published PF/ePF baselines on MATH500 with this same 1.5B model, within ~2 accuracy points, before trusting anything built on top.